# 05 Final Load Prep



**Target output:** `data/processed/burnout_sampled.csv`

**Final schema (matches Tableau dashboard):**

| Column | Type | Description |
|---|---|---|
| Academic Year | int | Year of study (1–4) |
| Age | int | Student age |
| Gender | string | Male / Female |
| Risk Level | string | Low / High |
| Academic Performance | float | Score 0–100 |
| Anxiety Score | float | Raw anxiety score 0–10 |
| Anxiety Norm | float | Normalised anxiety (0–1) |
| Count anxiety | int | 1 per row (for Tableau COUNT aggregation) |
| Burnout Score | float | Raw burnout score 0–10 |
| Depression Score | float | Raw depression score 0–10 |
| Depression Norm | float | Normalised depression (0–1) |
| Dropout Risk | float | Dropout risk score 0–10 |
| Exam Pressure | float | Exam pressure score 0–10 |
| Family Expectation | float | Family expectation score 0–10 |
| Financial Stress | float | Financial stress score 0–10 |
| Internet Usage | float | Internet usage hours |
| Mental Health Index | float | Mental health index 0–10 |
| Physical Activity | float | Physical activity score 0–10 |
| Physical Norm | float | Normalised physical activity (0–1) |
| Screen Time | float | Screen time hours |
| Screen time sectors | string | Low / Medium / High |
| Sleep Hours | float | Sleep hours per night |
| Sleep Hours (bin) | string | Binned sleep hours label |
| Sleep Norm | float | Normalised sleep hours (0–1) |
| Sleep sectors | string | Short / Normal / Long |
| Social Support | float | Social support score 0–10 |
| Stress Level | float | Stress level score 0–10 |
| Study Hours Per Day | float | Study hours per day |
| Study Hours Per Day (bin) | string | Binned study hours label |
| Support Norm | float | Normalised social support (0–1) |

## 5.1 Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH         = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'
TABLEAU_READY_PATH = PROJECT_ROOT / 'data/processed/burnout_sampled.csv'

df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

## 5.2 Rename Columns to Tableau-Friendly Names

Convert snake_case to Title Case with spaces to match the Tableau field names shown in the dashboard.

In [ ]:
RENAME_MAP = {
    'age':                  'Age',
    'gender':               'Gender',
    'academic_year':        'Academic Year',
    'study_hours_per_day':  'Study Hours Per Day',
    'exam_pressure':        'Exam Pressure',
    'academic_performance': 'Academic Performance',
    'stress_level':         'Stress Level',
    'anxiety_score':        'Anxiety Score',
    'depression_score':     'Depression Score',
    'sleep_hours':          'Sleep Hours',
    'physical_activity':    'Physical Activity',
    'social_support':       'Social Support',
    'screen_time':          'Screen Time',
    'internet_usage':       'Internet Usage',
    'financial_stress':     'Financial Stress',
    'family_expectation':   'Family Expectation',
    'burnout_score':        'Burnout Score',
    'mental_health_index':  'Mental Health Index',
    'risk_level':           'Risk Level',
    'dropout_risk':         'Dropout Risk',
}

df = df.rename(columns=RENAME_MAP)
print('Columns after rename:')
print(df.columns.tolist())

## 5.3 Derived Columns — Alias and Normalised Scores

Add `Anxiety` and `Depression` as direct aliases of their Score columns (required by Tableau schema),
then min-max normalise key psychological scores to a 0–1 scale.

In [ ]:
# Alias columns — Tableau expects both 'Anxiety' and 'Anxiety Score'
df['Anxiety']    = df['Anxiety Score']
df['Depression'] = df['Depression Score']

def min_max_norm(series: pd.Series) -> pd.Series:
    """Min-max normalise a series to [0, 1]."""
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(0.0, index=series.index)
    return ((series - lo) / (hi - lo)).round(4)

df['Anxiety Norm']    = min_max_norm(df['Anxiety Score'])
df['Depression Norm'] = min_max_norm(df['Depression Score'])
df['Physical Norm']   = min_max_norm(df['Physical Activity'])
df['Sleep Norm']      = min_max_norm(df['Sleep Hours'])
df['Support Norm']    = min_max_norm(df['Social Support'])

print('Alias + normalised columns added:')
check_cols = ['Anxiety', 'Depression', 'Anxiety Norm', 'Depression Norm',
              'Physical Norm', 'Sleep Norm', 'Support Norm']
df[check_cols].describe().round(4)

## 5.4 Derived Columns — Count Field

`Count anxiety` is a constant 1 per row, enabling Tableau to COUNT records in aggregations.

In [ ]:
df['Count anxiety'] = 1
print(f'Count anxiety column added. Unique values: {df["Count anxiety"].unique()}')

## 5.5 Derived Columns — Binned Fields

Create labelled bins for Sleep Hours, Study Hours Per Day, Screen Time, and Family Expectation.

In [ ]:
# ── Sleep Hours (bin) ──────────────────────────────────────────────────────────
sleep_bins   = [0, 4, 6, 7, 8, 10, 12]
sleep_labels = ['<4h', '4-6h', '6-7h', '7-8h', '8-10h', '10-12h']
df['Sleep Hours (bin)'] = pd.cut(
    df['Sleep Hours'],
    bins=sleep_bins,
    labels=sleep_labels,
    right=True,
    include_lowest=True
).astype(str)

print('Sleep Hours (bin):')
print(df['Sleep Hours (bin)'].value_counts().sort_index())

In [ ]:
# ── Study Hours Per Day (bin) ──────────────────────────────────────────────────
study_bins   = [0, 2, 4, 6, 8, 10, 24]
study_labels = ['0-2h', '2-4h', '4-6h', '6-8h', '8-10h', '10h+']
df['Study Hours Per Day (bin)'] = pd.cut(
    df['Study Hours Per Day'],
    bins=study_bins,
    labels=study_labels,
    right=True,
    include_lowest=True
).astype(str)

print('Study Hours Per Day (bin):')
print(df['Study Hours Per Day (bin)'].value_counts().sort_index())

In [ ]:
# ── Family Expectation (bin) ───────────────────────────────────────────────────
fe_bins   = [0, 2, 4, 6, 8, 10]
fe_labels = ['0-2', '2-4', '4-6', '6-8', '8-10']
df['Family Expectation (bin)'] = pd.cut(
    df['Family Expectation'],
    bins=fe_bins,
    labels=fe_labels,
    right=True,
    include_lowest=True
).astype(str)

print('Family Expectation (bin):')
print(df['Family Expectation (bin)'].value_counts().sort_index())

In [ ]:
# ── Screen time sectors ────────────────────────────────────────────────────────
# Low: 0–4h, Medium: 4–8h, High: 8h+
screen_bins   = [0, 4, 8, 16]
screen_labels = ['Low', 'Medium', 'High']
df['Screen time sectors'] = pd.cut(
    df['Screen Time'],
    bins=screen_bins,
    labels=screen_labels,
    right=True,
    include_lowest=True
).astype(str)

print('Screen time sectors:')
print(df['Screen time sectors'].value_counts())

In [ ]:
# ── Sleep sectors ──────────────────────────────────────────────────────────────
# Short: <6h, Normal: 6–8h, Long: >8h
def sleep_sector(hours):
    if hours < 6:    return 'Short'
    elif hours <= 8: return 'Normal'
    else:            return 'Long'

df['Sleep sectors'] = df['Sleep Hours'].apply(sleep_sector)

print('Sleep sectors:')
print(df['Sleep sectors'].value_counts())

## 5.6 KPI Verification

In [ ]:
print('=== KPI Summary ===')
print(f'Total students          : {len(df):,}')
print(f'High-risk students      : {(df["Risk Level"] == "High").sum():,} ({(df["Risk Level"] == "High").mean()*100:.1f}%)')
print(f'Mean Burnout Score      : {df["Burnout Score"].mean():.3f}')
print(f'Mean Mental Health Index: {df["Mental Health Index"].mean():.3f}')
print(f'Mean Dropout Risk       : {df["Dropout Risk"].mean():.3f}')
print(f'Mean Stress Level       : {df["Stress Level"].mean():.3f}')
print(f'Mean Anxiety Score      : {df["Anxiety Score"].mean():.3f}')
print(f'Mean Depression Score   : {df["Depression Score"].mean():.3f}')
print()
print('By Gender:')
print(df.groupby('Gender')[['Burnout Score', 'Dropout Risk', 'Mental Health Index']].mean().round(3))
print()
print('By Academic Year:')
print(df.groupby('Academic Year')[['Burnout Score', 'Dropout Risk', 'Stress Level']].mean().round(3))
print()
print('By Risk Level:')
print(df.groupby('Risk Level')[['Burnout Score', 'Dropout Risk', 'Mental Health Index']].mean().round(3))

## 5.7 Final Column Order and Schema Validation

In [ ]:
# Define the exact column order matching the Tableau schema
FINAL_COLUMNS = [
    # Dimensions
    'Academic Year',
    'Gender',
    'Risk Level',
    'Screen time sectors',
    'Sleep Hours (bin)',
    'Sleep sectors',
    'Study Hours Per Day (bin)',
    # Measures
    'Academic Performance',
    'Age',
    'Anxiety Norm',
    'Anxiety Score',
    'Burnout Score',
    'Count anxiety',
    'Depression Norm',
    'Depression Score',
    'Dropout Risk',
    'Exam Pressure',
    'Family Expectation',
    'Financial Stress',
    'Internet Usage',
    'Mental Health Index',
    'Physical Activity',
    'Physical Norm',
    'Screen Time',
    'Sleep Hours',
    'Sleep Norm',
    'Social Support',
    'Stress Level',
    'Study Hours Per Day',
    'Support Norm',
]

# Validate all columns exist
missing_cols = [c for c in FINAL_COLUMNS if c not in df.columns]
if missing_cols:
    print(f'WARNING — missing columns: {missing_cols}')
else:
    print('All required columns present.')

df_final = df[FINAL_COLUMNS].copy()
print(f'\nFinal shape: {df_final.shape}')
print(f'Columns: {df_final.columns.tolist()}')

In [ ]:
# Data types summary
print('Data types:')
print(df_final.dtypes.to_string())

In [ ]:
# Final null check
nulls = df_final.isnull().sum()
print(f'Total nulls: {nulls.sum()}')
if nulls.sum() > 0:
    print(nulls[nulls > 0])

In [ ]:
df_final.head(10)

## 5.8 Export Tableau-Ready Dataset

In [ ]:
TABLEAU_READY_PATH.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(TABLEAU_READY_PATH, index=False)

print(f'Saved Tableau-ready dataset to: {TABLEAU_READY_PATH}')
print(f'Rows   : {len(df_final):,}')
print(f'Columns: {len(df_final.columns)}')
print(f'File size: {TABLEAU_READY_PATH.stat().st_size / 1024 / 1024:.1f} MB')

## 5.9 Final Load Summary

| Derived Column | Logic | Purpose |
|---|---|---|
| Anxiety Norm | (Anxiety Score − min) / (max − min) | Normalised 0–1 for Tableau calculated fields |
| Depression Norm | (Depression Score − min) / (max − min) | Normalised 0–1 |
| Physical Norm | (Physical Activity − min) / (max − min) | Normalised 0–1 |
| Sleep Norm | (Sleep Hours − min) / (max − min) | Normalised 0–1 |
| Support Norm | (Social Support − min) / (max − min) | Normalised 0–1 |
| Count anxiety | Constant 1 | Enables COUNT aggregation in Tableau |
| Sleep Hours (bin) | pd.cut with 6 bins | Binned axis for Tableau bar charts |
| Study Hours Per Day (bin) | pd.cut with 6 bins | Binned axis for Tableau bar charts |
| Screen time sectors | Low / Medium / High (0–4 / 4–8 / 8+) | Categorical filter in Tableau |
| Sleep sectors | Short / Normal / Long (<6 / 6–8 / >8) | Categorical filter in Tableau |

**Output file:** `data/processed/burnout_sampled.csv`  
**Ready for:** Tableau Public dashboard import
